# 05 — Does Gaussian beat STE when the gradient passes through the WEIGHT quantizer?
**Re-upload the updated `qtta_lib.py`** (adds trainable fake-quantized weights). Adapts BN (lr 1e-3) plus conv weights (latent FP weights, 4/3-bit fake-quant forward, STE or Gaussian backward).
Groups: `BN` (reference), `BN+Wlast` (6 convs of the last stage), `BN+Wall` (all 18 block convs).

In [ ]:
import sys, os, glob, copy
import numpy as np, pandas as pd, torch, torch.nn as nn
sys.path.insert(0, os.path.dirname(glob.glob("/kaggle/input/**/qtta_lib.py", recursive=True)[0]))
import qtta_lib as Q
from qtta_lib import QCFG, QLayer
dev = "cuda" if torch.cuda.is_available() else "cpu"
CORR, SEEDS, SEV = ["gaussian_noise", "fog", "contrast"], [0, 1], 5
MEAN = torch.tensor(Q.MEAN).view(1, 3, 1, 1); STD = torch.tensor(Q.STD).view(1, 3, 1, 1)

def load_c(name, sev=SEV):
    path = glob.glob(f"/kaggle/input/**/{name}.npy", recursive=True)[0]
    lab = os.path.join(os.path.dirname(path), "labels.npy")
    sl = slice((sev - 1) * 10000, sev * 10000)
    x = torch.from_numpy(np.array(np.load(path, mmap_mode="r")[sl])).permute(0, 3, 1, 2).float() / 255
    return ((x - MEAN) / STD).to(dev), torch.from_numpy(np.load(lab)[sl]).long().to(dev)
DATA = {c: load_c(c) for c in CORR}

fp, _ = Q.load_fp32(Q.find_file("resnet20_fp32.pt"), dev)
train_loader, test_loader = Q.get_cifar10(bs=500)
MODELS = {n: Q.quantize_model(fp, wb, ab, train_loader, dev)[0] for n, (wb, ab) in {"W4A4": (4, 4), "W3A3": (3, 3)}.items()}

def entropy(lg): return -(lg.softmax(1) * lg.log_softmax(1)).sum(1).mean()

def setup(m, group):
    m.train()
    for p in m.parameters(): p.requires_grad_(False)
    bn = []
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.track_running_stats = False; mod.running_mean = None; mod.running_var = None
            mod.weight.requires_grad_(True); mod.bias.requires_grad_(True); bn += [mod.weight, mod.bias]
    wl = []
    if group != "BN":
        blocks = m.layers[6:9] if group == "BN+Wlast" else m.layers
        for b in blocks:
            for q in (b.conv1, b.conv2):
                q.enable_weight_training(); wl.append(q.w_lat)
    return bn, wl

def run_tta(model, x, y, group="BN", mode="ste", sigma=0.5, lr_bn=1e-3, lr_w=1e-4, bs=64, seed=0):
    m = copy.deepcopy(model); bn, wl = setup(m, group)
    QCFG.mode, QCFG.sigma = mode, sigma
    opt = torch.optim.Adam([{"params": bn, "lr": lr_bn}] + ([{"params": wl, "lr": lr_w}] if wl else []))
    perm = torch.randperm(len(x), generator=torch.Generator().manual_seed(seed)).to(dev)
    correct = 0
    for i in range(0, len(x), bs):
        idx = perm[i:i + bs]
        logits = m(x[idx])
        opt.zero_grad(set_to_none=True); entropy(logits).backward(); opt.step()
        correct += (logits.argmax(1) == y[idx]).sum().item()
    return 100 * correct / len(x)

## Sanity checks: weight-training path reproduces the PTQ model; weight gradients exist

In [ ]:
xb = DATA["fog"][0][:128]
for n, base in MODELS.items():
    ref = copy.deepcopy(base).eval()
    m = copy.deepcopy(base).eval()
    for b in m.layers:
        for q in (b.conv1, b.conv2): q.enable_weight_training()
    with torch.no_grad(): d = (ref(xb) - m(xb)).abs().max().item()
    assert d < 1e-4, (n, d)
    m2 = copy.deepcopy(base); bn, wl = setup(m2, "BN+Wall"); QCFG.mode = "ste"
    entropy(m2(xb)).backward()
    gn = [w.grad.norm().item() for w in wl]
    print(f"{n}: forward match |diff|={d:.1e} | weight-grad norms min/max = {min(gn):.3e}/{max(gn):.3e}")

## Experiment

In [ ]:
GROUPS, LRW = ["BN+Wlast", "BN+Wall"], [1e-4, 5e-4]
METHODS = [("STE", "ste", 0.0), ("g0.25", "gauss", 0.25), ("g0.5", "gauss", 0.5), ("g1.0", "gauss", 1.0)]
res = []
for mn, base in MODELS.items():
    for c in CORR:                                   # BN-only reference (STE), same seeds
        for seed in SEEDS:
            res.append(dict(model=mn, group="BN", lr_w=0, corruption=c, method="STE", seed=seed,
                            acc=run_tta(base, *DATA[c], "BN", "ste", seed=seed)))
    for g in GROUPS:
        for lw in LRW:
            for c in CORR:
                for name, mode, s in METHODS:
                    for seed in SEEDS:
                        res.append(dict(model=mn, group=g, lr_w=lw, corruption=c, method=name, seed=seed,
                                        acc=run_tta(base, *DATA[c], g, mode, s, lr_w=lw, seed=seed)))
            pd.DataFrame(res).to_csv("/kaggle/working/scan_weights.csv", index=False)
            print("done", mn, g, lw, flush=True)

In [ ]:
df = pd.DataFrame(res)
ref = df[df.group == "BN"].groupby("model").acc.mean()
print("BN-only STE reference:", ref.round(2).to_dict(), "\n")
d = df[df.group != "BN"]
T = d.groupby(["model", "group", "lr_w", "method"]).acc.mean().unstack("method")[[n for n, _, _ in METHODS]]
for n, _, _ in METHODS[1:]: T[f"d_{n}"] = T[n] - T["STE"]
print(T.round(2).to_string())
P = d.pivot_table(index=["model", "group", "lr_w", "corruption", "seed"], columns="method", values="acc")
print("\nwin rate vs STE:")
print(pd.DataFrame({n: (P[n] > P["STE"]).groupby(["model", "group", "lr_w"]).mean() for n, _, _ in METHODS[1:]}).round(2).to_string())